# EDA: Team Stats


## Purpose
Distribution of team-level features, era trends, correlation heatmap, and champion team profiles.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
from pathlib import Path
import sys; sys.path.insert(0, str(Path(".").parent / "src"))
from nba_predictor.config import cfg


## Load Team Features

In [ ]:
team_path = cfg.project_root / "data/processed/team_season_features.parquet"
if team_path.exists():
    df = pd.read_parquet(team_path)
    print(df.shape)
    df.head()
else:
    print("Run make process first")


## ORtg / DRtg Trends 1984–2025

In [ ]:
Path("../reports/figures").mkdir(parents=True, exist_ok=True)

# Coerce any object columns to numeric
for col in ["ORtg", "DRtg", "NRtg", "Pace", "3PAr", "eFG%"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

league_avg = df.groupby("season")[["ORtg", "DRtg", "NRtg", "Pace", "3PAr"]].mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
era_spans = [(1984, 1994, "Showtime", "#aec6cf"), (1995, 2004, "Defensive", "#ffcba4"),
             (2005, 2014, "Transition", "#b5ead7"), (2015, 2026, "Analytics", "#ffd1dc")]

ax = axes[0]
ax.plot(league_avg.index, league_avg["ORtg"], label="Avg ORtg", color="steelblue", lw=2)
ax.plot(league_avg.index, league_avg["DRtg"], label="Avg DRtg", color="tomato", lw=2)
for start, end, label, color in era_spans:
    ax.axvspan(start, end + 0.5, alpha=0.12, color=color)
    ax.text((start + end) / 2, 95, label, ha="center", fontsize=8, color="dimgray")
ax.axvline(2012, color="gray", ls="--", lw=1, alpha=0.7, label="Lockout '12")
ax.axvline(2020, color="purple", ls="--", lw=1, alpha=0.7, label="Bubble '20")
ax.set_ylabel("Points per 100 possessions")
ax.set_title("League-Average ORtg & DRtg by Season (1984–2026)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.bar(league_avg.index, league_avg["NRtg"], color="mediumseagreen", alpha=0.7)
ax2.axhline(0, color="black", lw=0.8)
ax2.set_ylabel("Net Rating"); ax2.set_xlabel("Season")
ax2.set_title("League Net Rating (≈ 0 each season — data quality check)")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../reports/figures/01_ortg_drtg_trends.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"ORtg range: {league_avg['ORtg'].min():.1f} – {league_avg['ORtg'].max():.1f}")
print(f"DRtg range: {league_avg['DRtg'].min():.1f} – {league_avg['DRtg'].max():.1f}")


## Pace Trend — The 3-Point Revolution

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
era_spans = [(1984, 1994, "Showtime", "#aec6cf"), (1995, 2004, "Defensive", "#ffcba4"),
             (2005, 2014, "Transition", "#b5ead7"), (2015, 2026, "Analytics", "#ffd1dc")]

for ax in axes:
    for start, end, _, color in era_spans:
        ax.axvspan(start, end + 0.5, alpha=0.12, color=color)
    ax.axvline(2012, color="gray", ls="--", lw=1, alpha=0.6)
    ax.axvline(2020, color="purple", ls="--", lw=1, alpha=0.6)
    ax.grid(alpha=0.3)

axes[0].plot(league_avg.index, league_avg["Pace"], color="darkorange", lw=2)
axes[0].set_ylabel("Pace (poss/48 min)")
axes[0].set_title("League-Average Pace — Slowdown then Revival")

axes[1].plot(league_avg.index, league_avg["3PAr"] * 100, color="mediumorchid", lw=2)
axes[1].set_ylabel("3PA Rate (%)")
axes[1].set_title("3-Point Attempt Rate — The Analytics Revolution")

# TS% proxy from eFG%
league_ts = df.groupby("season")["eFG%"].mean()
axes[2].plot(league_ts.index, league_ts * 100, color="teal", lw=2)
axes[2].set_ylabel("eFG% (%)")
axes[2].set_xlabel("Season")
axes[2].set_title("Effective FG% — Rising Efficiency")

plt.tight_layout()
plt.savefig("../reports/figures/01_pace_3pt_trends.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Pace 1984: {league_avg.loc[1984,'Pace']:.1f}  →  2026: {league_avg.loc[2026,'Pace']:.1f}")
print(f"3PA Rate 1984: {league_avg.loc[1984,'3PAr']:.3f}  →  2026: {league_avg.loc[2026,'3PAr']:.3f}")


## Champion Team Profiles

In [ ]:
series_path = cfg.project_root / "data/processed/series_dataset.parquet"
series_df = pd.read_parquet(series_path)

# Identify champions: team that won NBA Finals each season
finals = series_df[series_df["series_round"] == "Finals"].copy()
champions = set()
for _, row in finals.iterrows():
    champions.add((row["season"], row["series_winner"]))

# Tag rows in team features
df["is_champion"] = df.apply(lambda r: (r["season"], r["Team_abbrev"]) in champions, axis=1)
df["is_playoff"] = df["Playoff_experience_years"] > 0  # proxy — any postseason exp

compare_cols = ["NRtg_norm", "ORtg_norm", "DRtg_norm", "Win_pct", "SRS", "Playoff_experience_years"]
champ_df  = df[df["is_champion"]][compare_cols]
playoff_df = df[df["Win_pct"] > 0.5][compare_cols]  # rough playoff proxy for all seasons
other_df   = df[~df["is_champion"] & (df["Win_pct"] <= 0.5)][compare_cols]

summary = pd.DataFrame({
    "Champions":       champ_df.mean(),
    "Winning teams":   playoff_df.mean(),
    "Below .500":      other_df.mean(),
}).T

print(f"Champions identified: {len(champ_df)}")
print(summary.round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, col, title in zip(axes,
        ["NRtg_norm", "Win_pct", "Playoff_experience_years"],
        ["Net Rating (normalized)", "Win %", "Playoff Exp (prior 5yr)"]):
    data_champ   = champ_df[col].dropna()
    data_playoff = playoff_df[col].dropna()
    ax.hist(data_playoff, bins=20, alpha=0.5, color="steelblue", label="Winning teams", density=True)
    ax.hist(data_champ,   bins=15, alpha=0.7, color="gold",      label="Champions",    density=True)
    ax.axvline(data_champ.mean(),   color="goldenrod", lw=2, ls="--")
    ax.axvline(data_playoff.mean(), color="steelblue",  lw=2, ls="--")
    ax.set_title(title); ax.set_xlabel(col); ax.legend(fontsize=9)

plt.suptitle("Champion vs. Winning-Team Profiles", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("../reports/figures/01_champion_profiles.png", dpi=120, bbox_inches="tight")
plt.show()


## Feature Correlation Heatmap

In [ ]:
feat_cols = ["NRtg_norm", "ORtg_norm", "DRtg_norm", "Pace_norm",
             "eFG%_norm", "TOV%_norm", "ORB%_norm", "DRB%_norm",
             "Win_pct", "SRS_norm", "MOV_norm",
             "Playoff_experience_years", "Prior_playoff_win_pct", "L10_NRtg", "L10_Win_pct"]
corr = df[feat_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            annot_kws={"size": 8})
ax.set_title("Team Feature Correlation Matrix (lower triangle)", fontsize=13)
plt.tight_layout()
plt.savefig("../reports/figures/01_feature_correlation.png", dpi=120, bbox_inches="tight")
plt.show()

# Flag highly correlated pairs
high_corr = [(c1, c2, corr.loc[c1, c2])
             for i, c1 in enumerate(feat_cols)
             for j, c2 in enumerate(feat_cols)
             if i < j and abs(corr.loc[c1, c2]) > 0.7]
print(f"\nHighly correlated pairs (|r| > 0.7): {len(high_corr)}")
for c1, c2, r in sorted(high_corr, key=lambda x: -abs(x[2])):
    print(f"  {c1:20s} ↔ {c2:20s}  r={r:+.3f}")
